# Mutagenesis position coverage

What fraction of positions across the 40 mutagenesis regions have at least one
variant in the dataset used for model training/evaluation. Uses the same
filtering (`#_Plasmids > 2`, <3 space-separated variants) and Region/Location
extraction as `fig3_mutational_validation.ipynb`, and the same 40-region x
309-position (Location 2..310) space as that notebook.

In [11]:
import pandas as pd

INPUT_DIR = '../input_files'
N_REGIONS = 40
N_POSITIONS_PER_REGION = 310  # Location 2..310

df_mut_all = pd.read_csv(f'{INPUT_DIR}/Complete_Mut_Data_LibVariant30_Pscount5.csv')

df_mut = df_mut_all[df_mut_all['#_Plasmids'] > 2]
df_mut = df_mut[df_mut['Variants'].str.split(' ').apply(len) < 3]
print(len(df_mut), "variants with >2 plasmids and <3 mutations")

df_mut['Location'] = df_mut['Variants'].str.split(':').str[1]
df_mut = df_mut[df_mut['Location'] != 'WT']
df_mut['Location'] = df_mut['Location'].astype(int)

covered_positions = df_mut[['Region', 'Location']].drop_duplicates()

n_covered = len(covered_positions)
n_total = N_REGIONS * N_POSITIONS_PER_REGION
pct = 100 * n_covered / n_total

print(f"Regions in data: {df_mut['Region'].nunique()} (expected {N_REGIONS})")
print(f"Positions with >=1 variant: {n_covered} / {n_total} = {pct:.2f}%")

20057 variants with >2 plasmids and <3 mutations
Regions in data: 40 (expected 40)
Positions with >=1 variant: 9550 / 12400 = 77.02%


In [4]:
df_mut

,Region,Variants,#_Plasmids,DHT_Counts,LFC2WT_MLE,WT_seq,Lib,Mutated_seq,Location
0,overlapped_read_631,CGATCAGAGAAG:100:T>A,23,0.125000,-0.576534,GATCATGCTGAAATGCTTTTGAAGCTGCCTTTGTGCACCGTGGCGC...,2088,GATCATGCTGAAATGCTTTTGAAGCTGCCTTTGTGCACCGTGGCGC...,100
21,overlapped_read_631,CGATCAGAGAAG:100:T>C,21,0.369837,0.988429,GATCATGCTGAAATGCTTTTGAAGCTGCCTTTGTGCACCGTGGCGC...,1598,GATCATGCTGAAATGCTTTTGAAGCTGCCTTTGTGCACCGTGGCGC...,100
46,overlapped_read_631,CGATCAGAGAAG:100:T>G,3,0.123967,-0.588507,GATCATGCTGAAATGCTTTTGAAGCTGCCTTTGTGCACCGTGGCGC...,121,GATCATGCTGAAATGCTTTTGAAGCTGCCTTTGTGCACCGTGGCGC...,100
52,overlapped_read_631,CGATCAGAGAAG:101:T>A,10,0.108221,-0.784487,GATCATGCTGAAATGCTTTTGAAGCTGCCTTTGTGCACCGTGGCGC...,961,GATCATGCTGAAATGCTTTTGAAGCTGCCTTTGTGCACCGTGGCGC...,101
66,overlapped_read_631,CGATCAGAGAAG:101:T>C,31,0.190751,0.033232,GATCATGCTGAAATGCTTTTGAAGCTGCCTTTGTGCACCGTGGCGC...,2422,GATCATGCTGAAATGCTTTTGAAGCTGCCTTTGTGCACCGTGGCGC...,101
...,...,...,...,...,...,...,...,...,...
350611,overlapped_read_659,GGCCTGACAACA:98:A>G,30,0.655415,-0.126728,GCCTGGGCAACAAAGTGGGACCCTATCTGAAAAACAAAATGAAAAC...,2696,GCCTGGGCAACAAAGTGGGACCCTATCTGAAAAACAAAATGAAAAC...,98
350658,overlapped_read_659,GGCCTGACAACA:98:A>T,24,0.675542,-0.083094,GCCTGGGCAACAAAGTGGGACCCTATCTGAAAAACAAAATGAAAAC...,2216,GCCTGGGCAACAAAGTGGGACCCTATCTGAAAAACAAAATGAAAAC...,98
350702,overlapped_read_659,GGCCTGACAACA:99:A>C,3,0.650407,-0.137796,GCCTGGGCAACAAAGTGGGACCCTATCTGAAAAACAAAATGAAAAC...,123,GCCTGGGCAACAAAGTGGGACCCTATCTGAAAAACAAAATGAAAAC...,99
350710,overlapped_read_659,GGCCTGACAACA:99:A>G,23,0.779066,0.122607,GCCTGGGCAACAAAGTGGGACCCTATCTGAAAAACAAAATGAAAAC...,3019,GCCTGGGCAACAAAGTGGGACCCTATCTGAAAAACAAAATGAAAAC...,99


## Percentage of possible single-nucleotide variants observed

At each of the 309 positions (Location 2..310) in each of the 40 regions,
there are 3 possible single-nucleotide substitutions (4 bases - 1 reference),
so the full space is `40 * 309 * 3 = 37,080` possible single-point variants.
This checks how many of those are actually present in the quality-filtered
dataset (`#_Plasmids > 2`, single-mutation rows only).

In [13]:
N_SUBS_PER_POSITION = 3  # 4 bases - 1 reference
n_possible_single_variants = N_REGIONS * 310 * N_SUBS_PER_POSITION

n_split = df_mut_all[df_mut_all['#_Plasmids'] > 2]['Variants'].str.split(' ').apply(len)
single_variant_rows = df_mut_all[df_mut_all['#_Plasmids'] > 2][n_split == 2]  # length 1 = WT, length >=3 = multi-mutant combos

n_used = len(single_variant_rows)
pct_used = 100 * n_used / n_possible_single_variants

print(f"Possible single-nucleotide variants: {n_possible_single_variants} ({N_REGIONS} regions x 309 positions x {N_SUBS_PER_POSITION} substitutions)")
print(f"Single-point variants used (passing quality filter): {n_used}")
print(f"Percentage of possible variants used: {pct_used:.2f}%")

Possible single-nucleotide variants: 37200 (40 regions x 309 positions x 3 substitutions)
Single-point variants used (passing quality filter): 20017
Percentage of possible variants used: 53.81%
